In [1]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [2]:
class TrackedObject:
    def __init__(self, **kwargs):
        # history сам добавляем через super().__setattr__, чтобы не ловить в логе
        super().__setattr__('history', [])
        for name, value in kwargs.items():
            setattr(self, name, value)

    def __setattr__(self, name, value):
        before = len(self.__dict__)
        existed = name in self.__dict__
        super().__setattr__(name, value)
        after = len(self.__dict__)
        self.history.append({
            "op": "set",
            "name": name,
            "value": value,
            "existed_before": existed,
            "dict_size_before": before,
            "dict_size_after": after,
        })

    def __delattr__(self, name):
        before = len(self.__dict__)
        existed = name in self.__dict__
        old_value = self.__dict__.get(name, "<missing>")
        super().__delattr__(name)
        after = len(self.__dict__)
        self.history.append({
            "op": "del",
            "name": name,
            "old_value": old_value,
            "existed_before": existed,
            "dict_size_before": before,
            "dict_size_after": after,
        })

obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x

print(obj.__dict__)
for event in obj.history:
    print(event)


{'history': [{'op': 'set', 'name': 'x', 'value': 1, 'existed_before': False, 'dict_size_before': 1, 'dict_size_after': 2}, {'op': 'set', 'name': 'y', 'value': 2, 'existed_before': False, 'dict_size_before': 2, 'dict_size_after': 3}, {'op': 'set', 'name': 'z', 'value': 3, 'existed_before': False, 'dict_size_before': 3, 'dict_size_after': 4}, {'op': 'set', 'name': '__str__', 'value': <function <lambda> at 0x7ff3c64a1bf0>, 'existed_before': False, 'dict_size_before': 4, 'dict_size_after': 5}, {'op': 'del', 'name': 'x', 'old_value': 1, 'existed_before': True, 'dict_size_before': 5, 'dict_size_after': 4}], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x7ff3c64a1bf0>}
{'op': 'set', 'name': 'x', 'value': 1, 'existed_before': False, 'dict_size_before': 1, 'dict_size_after': 2}
{'op': 'set', 'name': 'y', 'value': 2, 'existed_before': False, 'dict_size_before': 2, 'dict_size_after': 3}
{'op': 'set', 'name': 'z', 'value': 3, 'existed_before': False, 'dict_size_before': 3, 'dict_size_after': 4

## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [3]:
def merge(seqs):
    seqs = [list(seq) for seq in seqs if seq]
    result = []

    while seqs:
        for seq in seqs:
            candidate = seq[0]
            if not any(candidate in other[1:] for other in seqs):
                break
        else:
            raise TypeError("Inconsistent hierarchy, cannot build C3 linearization")

        result.append(candidate)

        new_seqs = []
        for seq in seqs:
            if seq and seq[0] == candidate:
                seq.pop(0)
            if seq:
                new_seqs.append(seq)
        seqs = new_seqs

    return result

def c3_linearize(cls):
    if cls is object:
        return [object]
    bases = list(cls.__bases__)
    seqs = [c3_linearize(base) for base in bases] + [bases]
    return [cls] + merge(seqs)

class A:
    def who(self):
        return "A"

class B(A):
    def who(self):
        return "B"

class C(A):
    def who(self):
        return "C"

class D(B, C):
    pass

class E(C):
    def who(self):
        return "E"

class F(B, E):
    pass

class G(D, F):
    pass

for cls in (D, F, G):
    manual = [c.__name__ for c in c3_linearize(cls)]
    builtin = [c.__name__ for c in cls.__mro__]
    print(f"{cls.__name__}:")
    print("  c3_linearize ->", manual)
    print("  __mro__      ->", builtin)
    print("  who() result ->", cls().who())
    print()

print("Комментарий:")
print("- C3 сохраняет локальный порядок баз, указанный в объявлении класса.")
print("- Класс D ищет методы как D -> B -> C -> A -> object, поэтому берётся B.who.")
print("- Для G сначала идут его прямые базы D и F, затем общий B, потом E, затем C и A.")
print("- Один и тот же предок не дублируется, а порядок не противоречит MRO родителей.")


D:
  c3_linearize -> ['D', 'B', 'C', 'A', 'object']
  __mro__      -> ['D', 'B', 'C', 'A', 'object']
  who() result -> B

F:
  c3_linearize -> ['F', 'B', 'E', 'C', 'A', 'object']
  __mro__      -> ['F', 'B', 'E', 'C', 'A', 'object']
  who() result -> B

G:
  c3_linearize -> ['G', 'D', 'F', 'B', 'E', 'C', 'A', 'object']
  __mro__      -> ['G', 'D', 'F', 'B', 'E', 'C', 'A', 'object']
  who() result -> B

Комментарий:
- C3 сохраняет локальный порядок баз, указанный в объявлении класса.
- Класс D ищет методы как D -> B -> C -> A -> object, поэтому берётся B.who.
- Для G сначала идут его прямые базы D и F, затем общий B, потом E, затем C и A.
- Один и тот же предок не дублируется, а порядок не противоречит MRO родителей.


## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [4]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "base secret"
        self._semi_private = "base semi-private"
        self.public = "base public"

class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()
        self.__secret_value = "child secret"
        self._semi_private = "child semi-private"
        self.public = "child public"

    def dump_dict(self):
        return self.__dict__

base = SecureBase()
child = SecureChild()

print("Base __dict__:", base.__dict__)
print("Child __dict__:", child.dump_dict())
print()

print("Base dir (filtered):", [name for name in dir(base) if "secret" in name or "semi" in name or name == "public"])
print("Child dir (filtered):", [name for name in dir(child) if "secret" in name or "semi" in name or name == "public"])
print()

print("Stored names inside child.__dict__:")
for key, value in child.__dict__.items():
    print(f"  {key!r}: {value!r}")

print()
print("Access mangled from outside:")
print("base._SecureBase__secret_value =", base._SecureBase__secret_value)
print("child._SecureBase__secret_value =", child._SecureBase__secret_value)
print("child._SecureChild__secret_value =", child._SecureChild__secret_value)


Base __dict__: {'_SecureBase__secret_value': 'base secret', '_semi_private': 'base semi-private', 'public': 'base public'}
Child __dict__: {'_SecureBase__secret_value': 'base secret', '_semi_private': 'child semi-private', 'public': 'child public', '_SecureChild__secret_value': 'child secret'}

Base dir (filtered): ['_SecureBase__secret_value', '_semi_private', 'public']
Child dir (filtered): ['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private', 'public']

Stored names inside child.__dict__:
  '_SecureBase__secret_value': 'base secret'
  '_semi_private': 'child semi-private'
  'public': 'child public'
  '_SecureChild__secret_value': 'child secret'

Access mangled from outside:
base._SecureBase__secret_value = base secret
child._SecureBase__secret_value = base secret
child._SecureChild__secret_value = child secret


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [5]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")

    def __init__(self, x):
        self.x = x

def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d

n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

# Попытка динамического добавления атрибутов
for obj in (n, w, ww):
    try:
        obj.z = 999
        dynamic_result = "ok"
    except Exception as e:
        dynamic_result = f"{type(e).__name__}: {e}"
    print(type(obj).__name__, "dynamic z ->", dynamic_result)
    print(describe_instance(obj))
    print("contains '__dict__' in dir:", "__dict__" in dir(obj))
    print("contains '__weakref__' in dir:", "__weakref__" in dir(obj))
    print()

# weakref
for obj in (n, w, ww):
    try:
        r = weakref.ref(obj)
        weak_result = f"ok -> {r()}"
    except Exception as e:
        weak_result = f"{type(e).__name__}: {e}"
    print(type(obj).__name__, "weakref ->", weak_result)


NoSlots dynamic z -> ok
{'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y', 'z']}
contains '__dict__' in dir: True
contains '__weakref__' in dir: True

WithSlots dynamic z -> AttributeError: 'WithSlots' object has no attribute 'z' and no __dict__ for setting new attributes
{'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
contains '__dict__' in dir: False
contains '__weakref__' in dir: False

WithSlotsWeak dynamic z -> AttributeError: 'WithSlotsWeak' object has no attribute 'z' and no __dict__ for setting new attributes
{'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}
contains '__dict__' in dir: False
contains '__weakref__' in dir: True

NoSlots weakref -> ok -> <__main__.NoSlots object at 0x7ff3c62f45a0>
WithSlots weakref -> TypeError: cannot create weak reference to 'WithSlots' object

## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [6]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        self._weak_parent = weak_parent
        if weak_parent and parent is not None:
            self.parent = weakref.ref(parent)
        else:
            self.parent = parent

    def get_parent(self):
        if self._weak_parent and self.parent is not None:
            return self.parent()
        return self.parent

    def __repr__(self):
        parent = self.get_parent()
        parent_name = parent.name if parent is not None else None
        return f"Node(name={self.name!r}, parent={parent_name!r}, weak_parent={self._weak_parent})"

def refcount(obj):
    # getrefcount добавляет временную ссылку сам, поэтому вычитаем 1
    return sys.getrefcount(obj) - 1

# Цикл со сильными ссылками
a = Node("a")
b = Node("b", parent=a)
a.parent = b

wr_a = weakref.ref(a)
wr_b = weakref.ref(b)

print("Strong cycle objects:", a, b)
print("Strong cycle refcounts:", refcount(a), refcount(b))

del a, b
collected_strong = gc.collect()   # объекты должны быть собраны как циклический мусор

print("GC collected after strong cycle:", collected_strong)
print("wr_a after collect:", wr_a())
print("wr_b after collect:", wr_b())

print()

# Аналогичная структура, но одна из обратных ссылок - слабая
c = Node("c")
d = Node("d", parent=c, weak_parent=True)   # слабая ссылка d -> c
c.parent = d                                # сильная ссылка c -> d

print("Weak-linked objects:", c, d)
print("Weak-linked refcounts:", refcount(c), refcount(d))

wr_c = weakref.ref(c)
wr_d = weakref.ref(d)
print("wr_c before:", wr_c())
print("wr_d before:", wr_d())

del c
gc.collect()

print("wr_c after deletion:", wr_c())
print("d still alive:", wr_d())
print("d.get_parent() after c deleted:", wr_d().get_parent())

del d
gc.collect()
print("wr_d after deleting d:", wr_d())


Strong cycle objects: Node(name='a', parent='b', weak_parent=False) Node(name='b', parent='a', weak_parent=False)
Strong cycle refcounts: 3 3
GC collected after strong cycle: 8
wr_a after collect: None
wr_b after collect: None

Weak-linked objects: Node(name='c', parent='d', weak_parent=False) Node(name='d', parent='c', weak_parent=True)
Weak-linked refcounts: 2 3
wr_c before: Node(name='c', parent='d', weak_parent=False)
wr_d before: Node(name='d', parent='c', weak_parent=True)
wr_c after deletion: None
d still alive: Node(name='d', parent=None, weak_parent=True)
d.get_parent() after c deleted: None
wr_d after deleting d: None
